# Quasi-Steady Orientation Assumption

Sweeps over aspect ratios $\chi$ and initial angles to map translational ($\tau_t$) and rotational ($\tau_r$) relaxation times for ellipsoidal particles settling in a uniform flow.

### Import libraries & global style

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.ticker import EngFormatter
import sys
import os

os.makedirs("Media", exist_ok=True)

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except:
    plt.style.use("ggplot")

plt.rcParams.update({
    "font.family": "serif",
    "mathtext.fontset": "cm",
    "font.size": 24,
    "axes.titlesize": 24,
    "axes.labelsize": 24,
    "xtick.labelsize": 24,
    "ytick.labelsize": 24,
    "legend.fontsize": 20,
    "figure.titlesize": 24,
    "axes.linewidth": 1.5,
    "lines.linewidth": 2.2,
    "lines.markersize": 7,
})

### ⚙️ Simulation parameters

In [ ]:
# ── Time ──────────────────────────────────────────────────────────────────────
T  = 1.5          # total simulation time [s]
dt = 1e-4
t_eval = np.linspace(0, T, int(T / dt + 1))

# ── Particle & fluid properties ───────────────────────────────────────────────
d              = 150e-6   # equivalent-volume sphere diameter [m]
rho_p, rho_f   = 2000, 1.225    # particle / fluid density  [kg/m³]
mu_f           = 1.48e-5        # dynamic viscosity          [Pa·s]
g              = 0              # gravitational acceleration [m/s²]

# ── Free-stream ───────────────────────────────────────────────────────────────
U_inf, V_inf   = 0.01, 0.0
flow_angle     = np.arctan2(V_inf, U_inf)

# ── Sweep ranges ──────────────────────────────────────────────────────────────
chi_values = np.linspace(0.01, 0.99, 100)   # oblate \chi ∈ (0,1)
angles    = np.linspace(0, np.pi, 180)      # initial orientation [rad]

Re = rho_f * U_inf * d / mu_f
print(f"Particle Reynolds number  Re = {Re:.3f}")
print(f"chi sweep: {len(chi_values)} values,  angle sweep: {len(angles)} values")
print(f"Total integrations: {len(chi_values)*len(angles):,}")

### Geometry & Perrin friction

`Feq(chi)` returns the Perrin friction factor for an oblate spheroid with aspect ratio $\chi = b/a < 1$.

In [ ]:
def Feq(p):
    """Perrin friction factor for an oblate spheroid (chi = b/a < 1)."""
    if np.isclose(p, 1.0):
        return 1.0
    chi  = np.sqrt(1 - p**2) / p
    S   = 2 * np.arctan(chi) / chi
    num = (1 / p**2) - p**2
    den = 2 - S * (2 - (1 / p**2))
    return (4 / 3) * (num / den)

### Particle dynamics

State vector: $y = [\omega, \theta, u, v]$  
- $\omega$ — angular velocity [rad/s]  
- $\theta$ — orientation angle [rad]  
- $u, v$ — translational velocity components [m/s]

In [ ]:
def make_derivatives(chi, a, b, m, I, C_r, K0, K90):
    """Return the ODE RHS closure for a given aspect ratio."""
    V_sphere = np.pi * d**3 / 6
    A        = np.pi * d**2 / 4

    def get_derivatives(t, y):
        omega, theta, u, v = y
        u_rel  = u - U_inf
        v_rel  = v - V_inf
        V_mag  = max(np.hypot(u_rel, v_rel), 1e-8)
        phi    = theta - flow_angle
        Re_loc = rho_f * V_mag * d / mu_f

        Cd_0  = (24/Re_loc) * (K0
                 + 0.15 * chi**95.91  * Re_loc**0.687
                 + 0.2927 * (1-chi)**0.4374 * Re_loc**0.7512)
        Cd_90 = (24/Re_loc) * (K90
                 + 0.15 * chi**100.7  * Re_loc**0.687
                 + 0.1411 * (1-chi**24.75) * Re_loc**0.7143)

        sin_phi, cos_phi = np.sin(phi), np.cos(phi)
        Cd = Cd_0 + (Cd_90 - Cd_0) * sin_phi**2
        Cl = (Cd_0 - Cd_90
              + 2*(1/chi - 1)**0.542 / (1+chi)**7.85 * Re_loc**0.1516
              ) * sin_phi * cos_phi
        Ct = 1.85 * (1/chi - 1)**0.832 / Re_loc**0.146 * sin_phi * cos_phi

        F_mag  = 0.5 * rho_f * V_mag**2 * A
        drag_x = -Cd * F_mag * u_rel / V_mag
        drag_y = -Cd * F_mag * v_rel / V_mag
        lift_x = -Cl * F_mag * v_rel / V_mag
        lift_y =  Cl * F_mag * u_rel / V_mag

        tau    = -0.75 * Ct * rho_f * V_mag**2 * V_sphere - C_r * omega

        domega = tau / I
        dtheta = omega
        du     = (drag_x + lift_x) / m
        dv     = (drag_y + lift_y - m * g) / m
        return [domega, dtheta, du, dv]

    return get_derivatives

### Main sweep ($\chi$ $\times$ initial angle)

In [ ]:
t_phi_list     = []
t_v_list       = []
color_chi_list  = []
theta_list     = []

counter     = 0
total_steps = len(chi_values) * len(angles)

for chi in chi_values:

    # ── Per-chi geometry ───────────────────────────────────────────────────────
    V_sphere = np.pi * d**3 / 6
    a        = d / np.cbrt(8 * chi)
    b        = a * chi
    m        = rho_p * V_sphere
    I        = (1/5) * m * (a**2 + b**2)
    C_r      = np.pi * mu_f * d**3 * Feq(chi)

    # Drag anisotropy
    e     = np.sqrt(1 - chi**2)
    term1 = (8/3) / chi**(1/3)
    term2 = (2*chi/(1-chi**2)
             + 2*(1 - 2*chi**2)/(1-chi**2)**(3/2) * np.arctan(e/chi))
    term3 = ((-chi)/(1-chi**2)
             - (2*chi**2 - 3)/(1-chi**2)**(3/2) * np.arcsin(e))
    K0, K90 = term1 / term2, term1 / term3

    rhs = make_derivatives(chi, a, b, m, I, C_r, K0, K90)

    # ── Angle sweep ───────────────────────────────────────────────────────────
    for theta0 in angles:
        counter += 1
        pct    = min(1.0, counter / total_steps)
        filled = int(50 * pct)
        bar    = "█" * filled + "░" * (50 - filled)
        sys.stdout.write(f"\rchi = {chi:.2f} │{bar}│ {pct*100:5.1f}%")
        sys.stdout.flush()

        sol = solve_ivp(rhs, (0, T), [0.0, theta0, 0.0, 0.0],
                        t_eval=t_eval, method="RK45",
                        rtol=1e-3, atol=1e-6)

        t_arr     = sol.t
        omega_arr, theta_arr, u_arr, v_arr = sol.y
        phi_arr   = theta_arr - flow_angle

        # Translational settling time: V_mag first exceeds 0.99 U_inf
        V_mag_arr   = np.hypot(u_arr, v_arr)
        unsettled_v = np.where(V_mag_arr < 0.99 * U_inf)[0]
        t_v = (0.0 if len(unsettled_v) == 0
               else (t_arr[-1] if unsettled_v[-1] == len(t_arr)-1
                     else t_arr[unsettled_v[-1] + 1]))

        # Rotational settling time: phi within pi/200 of final value
        phi_inf     = phi_arr[-1]
        unsettled_r = np.where(np.abs(phi_arr - phi_inf) > np.pi/200)[0]
        t_phi = (0.0 if len(unsettled_r) == 0
                 else (t_arr[-1] if unsettled_r[-1] == len(t_arr)-1
                       else t_arr[unsettled_r[-1] + 1]))

        t_v_list.append(t_v)
        t_phi_list.append(t_phi)
        color_chi_list.append(chi)
        theta_list.append(theta0)

print("\nSweep complete.")

### Relaxation-time phase portrait

In [ ]:
plt.rcParams.update({
    "font.family": "serif",
    "mathtext.fontset": "cm",
    "font.size": 30,
    "axes.titlesize": 30,
    "axes.labelsize": 30,
    "xtick.labelsize": 30,
    "ytick.labelsize": 30,
    "legend.fontsize": 30,
    "figure.titlesize": 30,
    "axes.linewidth": 1.5,
    "lines.linewidth": 2.2,
})

# ── Arrays ────────────────────────────────────────────────────────────────────
t_phi_arr = np.array(t_phi_list)
t_v_arr   = np.array(t_v_list)
chi_arr    = np.array(color_chi_list)
theta_arr = np.array(theta_list)

# ── Marker sizes: largest at theta=0/pi, smallest at theta=pi/2 ───────────────
size_min, size_max = 10, 100
size_factor        = np.sin(theta_arr)**2
scaled_sizes       = size_min + (size_max - size_min) * size_factor

# Draw largest markers first (background), smallest on top
draw_order    = np.argsort(scaled_sizes)[::-1]
t_v_s         = t_v_arr[draw_order]
t_phi_s       = t_phi_arr[draw_order]
ar_s          = chi_arr[draw_order]
sizes_s       = scaled_sizes[draw_order]

# ── Figure ────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 13))

norm = mpl.colors.Normalize(vmin=0, vmax=1)
cmap = plt.cm.magma

sc = ax.scatter(
    t_v_s, t_phi_s,
    c=ar_s, cmap=cmap, norm=norm,
    s=sizes_s, marker="o",
    edgecolor="k", linewidth=0.2,
    alpha=1.0, zorder=3
)

# ── Axis limits & formatting ──────────────────────────────────────────────────
xmin, xmax = 0.15, 0.80
ymin, ymax = 1e-2, 0.60
ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)

eng_fmt = EngFormatter(unit="s")
ax.xaxis.set_major_formatter(eng_fmt)
ax.yaxis.set_major_formatter(eng_fmt)

ax.tick_params(width=1.5, length=8, pad=6)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# ── Diagonal τ_t = τ_r and shaded region ─────────────────────────────────────
x_fill = np.linspace(xmin, xmax, 300)
ax.fill_between(x_fill, x_fill, ymax,
                color="steelblue", alpha=0.12, zorder=0,
                label=r"$\tau_t \leq \tau_r$")
ax.plot([xmin, xmax], [xmin, xmax],
        color="black", linestyle="--", linewidth=1.5, alpha=0.5, zorder=1)

ax.grid(True, linestyle=":", linewidth=0.8, alpha=0.6, zorder=0)

# ── Axis labels ───────────────────────────────────────────────────────────────
ax.set_xlabel(r"Translational Relaxation Time, $\tau_t$", labelpad=10)
ax.set_ylabel(r"Rotational Relaxation Time, $\tau_r$", labelpad=10)

# ── Size legend (initial angle) ───────────────────────────────────────────────
dummy_angles = [0, np.pi/4, np.pi/2]
size_labels  = [r"$0$ or $\pi$", r"$\pi/4$ or $3\pi/4$", r"$\pi/2$"]
size_handles = [
    ax.scatter([], [], s=size_min + (size_max - size_min)*np.sin(a)**2,
               color="gray", edgecolor="k", linewidth=0.5)
    for a in dummy_angles
]
leg1 = ax.legend(
    size_handles, size_labels,
    title=r"Initial angle $\theta_0$",
    loc="upper left", bbox_to_anchor=(0.02, 0.98),
    framealpha=0.9, frameon=True,
    borderpad=0.8, handletextpad=0.6,
)
leg1.get_title().set_fontsize(24)
ax.add_artist(leg1)

# ── Region legend ─────────────────────────────────────────────────────────────
ax.legend(loc="upper left", bbox_to_anchor=(0.75, 1.00),
          framealpha=0.9, frameon=True, borderpad=0.18)

# ── Colorbar ──────────────────────────────────────────────────────────────────
cbar = fig.colorbar(sc, ax=ax, pad=0.02, aspect=30)
cbar.set_label(r"Aspect Ratio $\chi$", labelpad=12)
cbar.ax.tick_params(labelsize=24, width=1.2, length=6)
cbar.outline.set_linewidth(1.2)

plt.tight_layout()

Re_val = rho_f * U_inf * d / mu_f
out_path = f"Media/relaxation_times_Re={Re_val:.2f}.pdf"
plt.savefig(out_path, format="pdf", dpi=300, bbox_inches="tight")
print("Saved to", out_path)
plt.show()